# 随机装箱问题

**类别：** 装箱

来源: [https://www.hexaly.com/templates/stochastic-packing-problem](https://www.hexaly.com/templates/stochastic-packing-problem)


## 问题

在随机装箱问题中，需要将一组物品划分到若干箱子中。没有容量约束，每个箱子可以装入任何物品。该问题的随机性来源于物品的重量是随机的。在本例中，随机性通过不同的场景来表示，每个场景对应一组可能的物品重量。对于给定的物品到箱子的分配方案，每个场景都有一个与之对应的最大箱重。

如何选取最合适的目标函数以最小化这种随机的最大重量是一个挑战。事实上，最小化平均最大重量可能掩盖有风险的场景，而最小化最坏场景下的最大重量又可能过于悲观。一个常用的折中方案是对某个给定的百分位数进行优化，从而构建一个稳健的方案。因此，本例中我们采用的目标函数是最小化所有场景中最大重量的第 90 百分位数。

	

### 学到的建模原则

- 使用 [set 决策变量](https://www.hexaly.com/docs/last/mathematicaloperators/collectionvariables.html) 建模各箱子所装物品
- 定义 [lambda 函数](https://www.hexaly.com/docs/last/mathematicaloperators/delegates.html) 来计算每个箱子的总重量


## 数据

在本例中，我们在程序运行时随机生成实例。首先为每个物品选择一个均匀分布，然后对每个场景，从相应的均匀分布中独立采样得到各物品的重量。


## 程序

随机装箱问题的 Hexaly 模型使用 [set 决策变量](https://www.hexaly.com/docs/last/mathematicaloperators/collectionvariables.html) 来表示每个箱子中所装物品的集合。我们通过约束这些集合构成一个划分（partition），从而保证每个物品恰好被分配到一个箱子中。

我们使用对集合的可变参数 ‘sum（求和）’ 运算符以及一个返回该场景下指定物品索引对应重量的 [lambda 函数](https://www.hexaly.com/docs/last/mathematicaloperators/delegates.html) 来计算每个箱子在每个场景下的总重量。需要注意的是，该求和的项数在搜索过程中会随着集合大小变化而变化。然后我们可以计算每个场景对应的最大箱重。

最后，我们使用 ‘sort（排序）’ 运算符将最大重量数组按升序排序。这样便可以方便地取出目标函数的值，即排序后数组中的第 90 百分位数。


## Python 实现


In [ ]:
# Copyright (c) Hexaly. Permission is hereby granted to use, copy,
# and modify this code for applications developed with Hexaly.
from __future__ import print_function
import random
import math
import hexaly.optimizer


def generate_scenarios(nb_items, nb_scenarios, rng_seed):
    random.seed(rng_seed)

    # Pick random parameters for each item distribution
    items_dist = []
    for _ in range(nb_items):
        item_min = random.randint(10, 100)
        item_max = item_min + random.randint(0, 50)
        items_dist.append((item_min, item_max))

    # Sample the distributions to generate the scenarios
    scenario_item_weights = [[random.randint(*dist) for dist in items_dist]
                               for _ in range(nb_scenarios)]
    return scenario_item_weights


def main(nb_items, nb_bins, nb_scenarios, seed, time_limit):
    # Generate instance data
    scenario_item_weights_data = generate_scenarios(nb_items, nb_scenarios, seed)

    with hexaly.optimizer.HexalyOptimizer() as optimizer:
        #
        # Declare the optimization model
        #
        model = optimizer.model

        # Set decisions: bins[k] represents the items in bin k
        bins = [model.set(nb_items) for _ in range(nb_bins)]

        # Each item must be in one bin and one bin only
        model.constraint(model.partition(bins))

        scenarios_item_weights = model.array(scenario_item_weights_data)

        # Compute max weight for each scenario
        scenarios_max_weights = model.array(
            model.max(
                model.sum(bin,
                          model.lambda_function(
                              lambda i:
                                  model.at(scenarios_item_weights, k, i)))
                for bin in bins) for k in range(nb_scenarios))

        # Compute the 9th decile of scenario max weights
        stochastic_max_weight = \
            model.sort(scenarios_max_weights)[int(math.ceil(0.9 * (nb_scenarios - 1)))]

        model.minimize(stochastic_max_weight)
        model.close()

        # Parameterize the optimizer
        optimizer.param.time_limit = time_limit

        optimizer.solve()

        #
        # Write the solution
        #
        print()
        print("Scenario item weights:")
        for i, scenario in enumerate(scenario_item_weights_data):
            print(i, ': ', scenario, sep='')

        print()
        print("Bins:")
        for k, bin in enumerate(bins):
            print(k, ': ', bin.value, sep='')


if __name__ == '__main__':
    nb_items = 10
    nb_bins = 2
    nb_scenarios = 3
    rng_seed = 42
    time_limit = 2

    main(
        nb_items,
        nb_bins,
        nb_scenarios,
        rng_seed,
        time_limit
    )
